# Lab 15 - Human Activity Recognition using LSTM
Esraaj Sarkar Gupta, on the 10th of May, 2026

In [4]:
# ---- Imports and Helpers ---- #
import os
import numpy as np
from numpy import mean, std, dstack
from pandas import read_csv
from matplotlib import pyplot as plt

from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout, LSTM
from keras.utils import to_categorical

print(f"Completed all imports successfully...")

# Is this the last lab already??
# I will miss talking to you every week ike this, Indrani Ma'am.
# I hope you TA another course for me in the future where I can can submit labs to you again!

Completed all imports successfully...


In [ ]:
# ---- Dataset Loading Functions ---- #

def load_file(filepath):
    """Load a single file as a numpy array."""
    dataframe = read_csv(filepath, header=None, sep="\s+") # Whitespace arg is deprecated
    return dataframe.values

def load_group(filenames, prefix=''):
    """Load a list of files into a 3D array of [samples, timesteps, features]."""
    loaded : list = list([])

    for name in filenames:
        data = load_file(prefix + name)
        loaded.append(data)

    # stack group so that features are in 3D
    loaded = dstack(loaded)
    return loaded

def load_dataset_group(group, prefix=''):
    """Load a dataset group, such as train or test."""

    filepath = prefix + group + '/Inertial Signals/'

    # Load all the files as a single array
    filenames = list([])

    # Total acceleration
    filenames += ['total_acc_x_'+group+'.txt', 'total_acc_y_'+group+'.txt', 'total_acc_z_'+group+'.txt']
    
    # Body acceleration
    filenames += ['body_acc_x_'+group+'.txt', 'body_acc_y_'+group+'.txt', 'body_acc_z_'+group+'.txt']
    
    # Body gyroscope
    filenames += ['body_gyro_x_'+group+'.txt', 'body_gyro_y_'+group+'.txt', 'body_gyro_z_'+group+'.txt']
    
    # load input data
    X = load_group(filenames, filepath)
    # load class output
    y = load_file(prefix + group + '/y_'+group+'.txt')
    return X, y

def load_dataset(prefix=''):
    """Load the dataset, returns train and test X and y elements."""
    
    # Training dataset
    trainX, trainy = load_dataset_group('train', prefix + 'UCI HAR Dataset/')
    
    # Testing dataset
    testX, testy = load_dataset_group('test', prefix + 'UCI HAR Dataset/')
    
    # Zero-offset class values
    trainy = trainy - 1
    testy = testy - 1
    
    # One-Hot encoding
    trainy = to_categorical(trainy)
    testy = to_categorical(testy)

    print(f"Dataset loaded. Train shape: {trainX.shape}, Test shape: {testX.shape}")
    
    return trainX, trainy, testX, testy

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_274395/3928900262.py:5: SyntaxWarning: invalid escape sequence '\s'
  dataframe = read_csv(filepath, header=None, sep="\s+")


In [ ]:
# ---- Model Evaluation & 10-Iteration Run ---- #
def evaluate_model(trainX, trainy, testX, testy):
    """
    Define, compile, fit, and evaluate the LSTM model.
    """
    
    # -- Run Settings -- #
    verbose = 1     # I need to know what's going on for 30 minutes in this machine
    epochs = 15     # I only have 30 minutes
    batch_size = 64

    # Setting dimensions
    n_timesteps, n_features, n_outputs = trainX.shape[1], trainX.shape[2], trainy.shape[1]
    
    # Define Model
    model = Sequential()
    model.add(LSTM(100, input_shape=(n_timesteps, n_features)))
    model.add(Dropout(0.5))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(n_outputs, activation='softmax'))
    
    # Compile Model
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    # Fit Model (as the function name would suggest)
    model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose)
    
    # Evaluate model
    _, accuracy = model.evaluate(testX, testy, batch_size=batch_size, verbose=0)
    
    return accuracy

def run_experiment(repeats=10):
    """
    Run the model evaluation multiple times and report results.
    """
    trainX, trainy, testX, testy = load_dataset()
    
    scores : list= list([])
    for r in range(repeats):
        print(f'\n--- Starting Run {r+1} ---')

        score = evaluate_model(trainX, trainy, testX, testy)
        score = score * 100.0

        print(f'> Run {r+1} Complete: Accuracy = {score:.3f}%')
        
        scores.append(score)
        
    print(f'\nFinal Results over {repeats} runs:')
    print(f'Mean Accuracy: {mean(scores):.3f}%')
    print(f'Standard Deviation: {std(scores):.3f}')

# Execute the 10 runs
run_experiment()

# The output will be a little long, please bear with me.

Dataset loaded. Train shape: (7352, 128, 9), Test shape: (2947, 128, 9)

--- Starting Run 1 ---
Epoch 1/15


/home/esraaj/jupyterenv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.4823 - loss: 1.2136
Epoch 2/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.6459 - loss: 0.7935
Epoch 3/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.7021 - loss: 0.7548
Epoch 4/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.6926 - loss: 0.7967
Epoch 5/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.7530 - loss: 0.6066
Epoch 6/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.8327 - loss: 0.4316
Epoch 7/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - accuracy: 0.8995 - loss: 0.2944
Epoch 8/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.9236 - loss: 0.2197
Epoch 9/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.9193 - loss: 0.2229
Epoch 10/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.9272 - loss: 0.1975
Epoch 11/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.9392 - loss: 0.1635
Epoch 12/15
115/115 ━━━━━━━━━━━━━━━━━━━━ 9s 75ms/ste

# Report

`` Max CPU Temp 96C``<br>
`` Runtime 24m 36s``

#### Question 1

Human Activity Recognition uses continuous sensor data collected over time. Sequential time series data such as this show strong temporal dependencies between measurements. For this reason, we use LSTMs that, unlike SVMs or Random Forest Classifiers, do not treat each measurement as an independent data point. LSTMs are capable of learning from sequential, temporal dynamics.

#### Question 2
This is a many-to-one scenario, since many time steps (128 of them from 3D sensor data) are used to output a single classification to represent the state of the subject during the period of data collection.

#### Question 3

In the context of LSTMs, the Long-Term Memory is represented by the cell state $C_t$. The cell state is a vector that represents the internal memory of the unit. It is modified only by linear interactions (such as multiplication and addition) through the forget and input gates. Since the gradient may flow through this state with very little modification, information from many time steps is preserved. This allows the preservation of a stable representation of the global context; hence "long-term" memory.
<br>
On the other hand, Short-Term Memoryis given by the hidden state $h_t$. It is calculated by passing the current cell state through a non-linear activation function (such as tanh) and filtering it with the output gate. This contains information relevant to the local context and used to compute the prediction. Unlike the cell state, the hidden state is voltatile and changes significantly at every ste of the sequence.

#### Question 4
An LSTM node has three primary parts or "gates".

* The Forget Gate: This gate decides which information from the previous long-gterm memory (cell state) should be discarded or kept.It takes the current input and the previous short-term memory (hidden state) and outputs a value between 0 and 1 for each number in the cell state as a binary decision for retention or discarding.

* Input Gate: This gate determines what new information from the current time step is relevant and should be added onto the long-term memory. It involves a two-step process wehre a sigmoid layer decides which values to update and a non-linear tanh layer creates a vector of new candidate values that could be added to the cell state.

* Output Gate: This gate determines the next short-term memory (hidden state). It filters the newly updated cell state based on the current input and the previous hidden state to ensure only the relevant information is passed on to the next node and used for the final prediction.

#### Question 5

Traditional RNNs suffer heavily from the vanishing gradient problem. When dealing with long data sequences, the error gradients used to update the weights shrink exponentially as they propagate backward through time. This makes it almost impossible for standard RNNs to learn long-term dependencies. LSTMs fix this via the cell state and the gating mechanisms. This allows error signals to flow relatively unchanged over multiple time steps, effectively bridging long temporal gaps.